# Vector stores and semantic search



In [15]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import numpy as np

## Part I: Basic vector store implementation

In [16]:
import numpy as np

class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents = []
        self.embeddings = []

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        new_embs = self.model.encode([doc.text for doc in documents], convert_to_numpy=True)
        self.embeddings.extend(new_embs)

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if not self.documents:
            return []
        query_emb = self.model.encode([query], convert_to_numpy=True)
        scores = util.cos_sim(query_emb, self.embeddings)[0].numpy()
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [SearchResult(float(scores[i]), self.documents[i]) for i in top_idx]

In [17]:
df = pd.read_csv("Animal_Fun_Facts/animal-fun-facts-dataset.csv").fillna("")

documents = [
    Document(
        text=row["text"],
        metadata={
            "animal_name": row["animal_name"],
            "source":       row["source"],
            "media_link":   row["media_link"],
            "wikipedia_link": row["wikipedia_link"],
        }
    )
    for _, row in df.iterrows()
    if row["text"].strip()       
]

In [18]:
model = SentenceTransformer("all-MiniLM-L6-v2")
store = VectorStore(model)
store.add_documents(documents)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8906.57it/s]


In [19]:
queries = [
    "are the only mammals with wings",
    "mammals that can fly",
    "16 hours a day sleeping",
    "tongue is about 18 inches long",
    "reptiles and cold-blooded creatures",
]

In [20]:
def run_queries(queries: list[str], top_k: int = 5):
    for q in queries:
        print(f"\n{'='*60}\nQuery: {q}\n{'='*60}")

        for i, r in enumerate(store.search(q, top_k=top_k), 1):
            print(f"\n[{i}] Score: {r.score:.4f}")
            print(f"Text: {r.document.text[:120].strip()}...")
            for k, v in r.document.metadata.items():
                if v:
                    print(f"{k}: {v}")

run_queries(queries)


Query: are the only mammals with wings

[1] Score: 0.8304
Text: Bats are the only mammals with wings, and the only ones that can truly fly...
animal_name: bat
source: https://www.animalfactsencyclopedia.com/Bat-facts.html
wikipedia_link: /wiki/Bat

[2] Score: 0.7139
Text: Bats are the world's only flying mammals. Other mammals may glide through the air, but bats flap their wings and fly....
animal_name: malayan flying fox
source: https://seaworld.org/animals/facts/mammals/malayan-flying-fox/
wikipedia_link: /wiki/Large_flying_fox

[3] Score: 0.6941
Text: They don’t fly, they glide.
The only mammal which can independently fly is the bat. Instead, colugas glide which works i...
animal_name: colugo (flying lemur)
source: https://factanimal.com/colugo/
wikipedia_link: /wiki/Colugo

[4] Score: 0.6743
Text: Bats are the only flying mammals and comprise the second largest order of mammals in the world....
animal_name: bats
source: https://seaworld.org/animals/facts/mammals/bats/
wikipedia_li

## Part II: Filtering by metadata

In [21]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        pass

    def add_documents(self, documents: list[Document]):
        pass

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        pass